In [8]:
import cirq
import numpy as np

In [9]:
# Define the oracle for Grover's algorithm
def create_grover_oracle(qubits, marked_item):
    oracle = cirq.Circuit()
    if marked_item < len(qubits):
        oracle.append(cirq.X(qubits[marked_item]))  # Mark the item with a NOT gate
        oracle.append(cirq.H.on_each(*qubits))
        oracle.append(cirq.Z(qubits[-1]).controlled_by(*qubits[:-1]))
        oracle.append(cirq.H.on_each(*qubits))
        oracle.append(cirq.X(qubits[marked_item]))  # Undo the NOT gate
    return oracle

In [10]:
# Define the diffusion operator for Grover's algorithm
def create_diffusion_operator(qubits):
    diffusion = cirq.Circuit()
    diffusion.append(cirq.H.on_each(*qubits))
    diffusion.append(cirq.X.on_each(*qubits))
    diffusion.append(cirq.Z(qubits[-1]).controlled_by(*qubits[:-1]))
    diffusion.append(cirq.X.on_each(*qubits))
    diffusion.append(cirq.H.on_each(*qubits))
    return diffusion

In [11]:
# Define Grover's algorithm circuit
def create_grover_circuit(qubits, oracle, diffusion, num_iterations):
    grover_circuit = cirq.Circuit()
    
    # Initialize qubits in a uniform superposition
    grover_circuit.append(cirq.H.on_each(*qubits))
    
    # Apply Grover's iterations
    for _ in range(num_iterations):
        grover_circuit.append(oracle)
        grover_circuit.append(diffusion)
    
    return grover_circuit

In [12]:
# Simulate the quantum circuit
def run_simulation(grover_circuit):
    simulator = cirq.Simulator()
    result = simulator.simulate(grover_circuit)
    return result

In [37]:
# Find the marked item using Grover's algorithm
def find_marked_item(num_qubits, marked_item):
    qubits = cirq.LineQubit.range(num_qubits)
    
    oracle = create_grover_oracle(qubits, marked_item)
    diffusion = create_diffusion_operator(qubits)
    
    # Set the number of iterations for Grover's algorithm
    num_iterations = int(np.pi / 4 * np.sqrt(2**num_qubits))
    
    grover_circuit = create_grover_circuit(qubits, oracle, diffusion, num_iterations)
    
    # Simulate the quantum circuit
    result = run_simulation(grover_circuit)
    
    # Measure the qubits to get the final result
    measurements = result.measurements

    # Convert qubits to keys used in measurements
    qubit_keys = tuple(qubit.x for qubit in qubits)

    # Get the result for the first qubit as an example
    final_state = measurements[qubit_keys]

    # Convert final state to decimal to get the marked item
    marked_item_found = int(''.join(map(str, final_state.flatten())), 2)
    
    return marked_item_found

In [38]:
# Example usage
if __name__ == "__main__":
    num_qubits = 3
    marked_item = 3  # Example: The item we want to find in the unsorted list
    
    found_item = find_marked_item(num_qubits, marked_item)
    
    print(f"Marked item found at index: {found_item}")

KeyError: (0, 1, 2)